<a href="https://colab.research.google.com/github/GopalKrishna-India/Geospatial/blob/master/Python_TreeMapping_3Band_HighResolutionData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# TREE COVER MAPPING IN GOOGLE COLAB
# Satellite image + Meta Canopy Height + Random Forest
# ================================================================

# ------------------------------------------------
# 0. INSTALL REQUIRED PACKAGES
# ------------------------------------------------

!pip -q install earthengine-api geemap rasterio scikit-learn matplotlib


# ------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------

import os
import ee
import geemap
import rasterio
import numpy as np
import matplotlib.pyplot as plt

from rasterio.warp import reproject, Resampling, transform_bounds
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, classification_report


# ------------------------------------------------
# 2. INITIALIZE GOOGLE EARTH ENGINE
# ------------------------------------------------

try:
    ee.Initialize(project='ee-geoinformers')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='ee-geoinformers')

print("Earth Engine initialized.")
print("Project: ee-geoinformers")


# ------------------------------------------------
# 3. FILE PATHS
# ------------------------------------------------

SATELLITE_FILE = "/content/satellite_clip1.tif"

CANOPY_FILE = "/content/canopy_height.tif"

OUTPUT_TREE = "/content/tree_cover_classified.tif"

OUTPUT_TRAINING = "/content/training_samples.tif"


# ------------------------------------------------
# 4. CHECK SATELLITE IMAGE
# ------------------------------------------------

if not os.path.exists(SATELLITE_FILE):

    raise FileNotFoundError(
        "Satellite image not found:\n" + SATELLITE_FILE
    )


with rasterio.open(SATELLITE_FILE) as src:

    satellite = src.read().astype("float32")

    sat_profile = src.profile.copy()

    sat_transform = src.transform
    sat_crs = src.crs

    sat_height = src.height
    sat_width = src.width

    sat_bounds = src.bounds

    sat_nbands = src.count


print("\n================ SATELLITE IMAGE ================")

print("File       :", SATELLITE_FILE)
print("Bands      :", sat_nbands)
print("Width      :", sat_width)
print("Height     :", sat_height)
print("CRS        :", sat_crs)
print("Resolution :", sat_transform.a, abs(sat_transform.e))
print("Bounds     :", sat_bounds)


if sat_nbands < 3:

    raise ValueError(
        "The satellite image must contain at least 3 bands."
    )


# ------------------------------------------------
# 5. CREATE EARTH ENGINE REGION
# ------------------------------------------------

left, bottom, right, top = transform_bounds(
    sat_crs,
    "EPSG:4326",
    sat_bounds.left,
    sat_bounds.bottom,
    sat_bounds.right,
    sat_bounds.top
)

region = ee.Geometry.Rectangle(
    [left, bottom, right, top],
    proj="EPSG:4326",
    geodesic=False
)

print("\nGEE region created.")


# ------------------------------------------------
# 6. LOAD META CANOPY HEIGHT
#    SAME DATASET USED IN YOUR GEE WORKFLOW
# ------------------------------------------------

canopy_ht = ee.ImageCollection(
    "projects/sat-io/open-datasets/facebook/meta-canopy-height"
)

print("Meta Canopy Height collection loaded.")


# Mosaic collection
canopy_image = canopy_ht.mosaic().clip(region)


# ------------------------------------------------
# 7. EXPORT CANOPY HEIGHT FROM GEE TO COLAB
# ------------------------------------------------

print("\nDownloading canopy height from GEE...")

geemap.ee_export_image(
    canopy_image,
    filename=CANOPY_FILE,
    scale=10,
    region=region,
    file_per_band=False
)

print("Canopy height saved:")
print(CANOPY_FILE)


# ------------------------------------------------
# 8. READ CANOPY HEIGHT
# ------------------------------------------------

with rasterio.open(CANOPY_FILE) as src:

    canopy_raw = src.read(1).astype("float32")

    canopy_raw_transform = src.transform
    canopy_raw_crs = src.crs


print("\n=============== CANOPY HEIGHT ===============")

print("Shape :", canopy_raw.shape)
print("CRS   :", canopy_raw_crs)

valid_ch = np.isfinite(canopy_raw)

if valid_ch.any():

    print(
        "Min canopy height :",
        np.nanmin(canopy_raw[valid_ch])
    )

    print(
        "Max canopy height :",
        np.nanmax(canopy_raw[valid_ch])
    )


# ------------------------------------------------
# 9. RESAMPLE CANOPY HEIGHT TO SATELLITE GRID
# ------------------------------------------------
#
# This is important.
#
# The satellite image and canopy-height image may have
# different dimensions/resolutions.
#
# We therefore force canopy height onto EXACTLY the
# satellite image grid.
# ------------------------------------------------

canopy = np.zeros(
    (sat_height, sat_width),
    dtype="float32"
)


reproject(
    source=canopy_raw,
    destination=canopy,

    src_transform=canopy_raw_transform,
    src_crs=canopy_raw_crs,

    dst_transform=sat_transform,
    dst_crs=sat_crs,

    resampling=Resampling.bilinear
)


print("\nCanopy height aligned to satellite image.")

print("Satellite :", satellite.shape)
print("Canopy    :", canopy.shape)


# ------------------------------------------------
# 10. BUILD TREE / NON-TREE REFERENCE MASK
# ------------------------------------------------
#
# Same basic canopy-height logic:
#
# CH >= 4 m  -> TREE
# CH <  4 m  -> NON-TREE
#
# Invalid canopy pixels are excluded.
# ------------------------------------------------

TREE_HEIGHT_THRESHOLD = 4.0


valid = np.isfinite(canopy)


tree_reference = (
    (canopy >= TREE_HEIGHT_THRESHOLD) &
    valid
)


non_tree_reference = (
    (canopy < TREE_HEIGHT_THRESHOLD) &
    valid
)


print("\n=============== REFERENCE MASK ===============")

print(
    "Tree reference pixels    :",
    np.sum(tree_reference)
)

print(
    "Non-tree reference pixels:",
    np.sum(non_tree_reference)
)


# ------------------------------------------------
# 11. PREPARE SATELLITE FEATURES
# ------------------------------------------------
#
# We use the first three bands of satellite_clip1.tif.
#
# IMPORTANT:
# This assumes the three bands are already the desired
# predictor bands.
# ------------------------------------------------

features = satellite[:3]

print("\nSatellite predictor shape:")
print(features.shape)


# ------------------------------------------------
# 12. REMOVE INVALID SATELLITE PIXELS
# ------------------------------------------------

valid_satellite = np.all(
    np.isfinite(features),
    axis=0
)

# Also remove pixels where all three bands are zero
nonzero_satellite = np.any(
    features != 0,
    axis=0
)

valid_pixels = (
    valid_satellite &
    nonzero_satellite
)


# ------------------------------------------------
# 13. CREATE TREE / NON-TREE SAMPLE LOCATIONS
# ------------------------------------------------

tree_locations = np.where(
    tree_reference & valid_pixels
)

non_tree_locations = np.where(
    non_tree_reference & valid_pixels
)


print("\nAvailable reference pixels:")

print(
    "Tree    :",
    len(tree_locations[0])
)

print(
    "Non-tree:",
    len(non_tree_locations[0])
)


# ------------------------------------------------
# 14. BALANCED RANDOM SAMPLING
# ------------------------------------------------

SAMPLE_SIZE = 5000

rng = np.random.default_rng(42)


tree_count = len(tree_locations[0])
non_tree_count = len(non_tree_locations[0])


if tree_count == 0:

    raise ValueError(
        "No tree reference pixels found. "
        "Check canopy height dataset, projection and extent."
    )


if non_tree_count == 0:

    raise ValueError(
        "No non-tree reference pixels found."
    )


sample_n = min(
    SAMPLE_SIZE,
    tree_count,
    non_tree_count
)


tree_idx = rng.choice(
    tree_count,
    size=sample_n,
    replace=False
)


non_tree_idx = rng.choice(
    non_tree_count,
    size=sample_n,
    replace=False
)


tree_rows = tree_locations[0][tree_idx]
tree_cols = tree_locations[1][tree_idx]


non_tree_rows = non_tree_locations[0][non_tree_idx]
non_tree_cols = non_tree_locations[1][non_tree_idx]


# ------------------------------------------------
# 15. EXTRACT TRAINING VALUES
# ------------------------------------------------

tree_values = features[
    :,
    tree_rows,
    tree_cols
].T


non_tree_values = features[
    :,
    non_tree_rows,
    non_tree_cols
].T


X = np.vstack([
    tree_values,
    non_tree_values
])


y = np.concatenate([
    np.ones(sample_n, dtype=np.uint8),
    np.zeros(sample_n, dtype=np.uint8)
])


print("\n=============== TRAINING DATA ===============")

print("Tree samples    :", sample_n)
print("Non-tree samples:", sample_n)

print("X shape:", X.shape)
print("Y shape:", y.shape)


# ------------------------------------------------
# 16. REMOVE INVALID TRAINING VALUES
# ------------------------------------------------

good_training = np.all(
    np.isfinite(X),
    axis=1
)


X = X[good_training]
y = y[good_training]


print(
    "Valid training samples:",
    len(y)
)


# ------------------------------------------------
# 17. TRAIN / VALIDATION SPLIT
# ------------------------------------------------

from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)


print("\nTraining:", len(y_train))
print("Testing :", len(y_test))


# ------------------------------------------------
# 18. RANDOM FOREST
# ------------------------------------------------

rf = RandomForestClassifier(

    n_estimators=500,

    min_samples_leaf=3,

    max_features="sqrt",

    max_samples=0.7,

    random_state=42,

    n_jobs=-1
)


print("\nTraining Random Forest...")

rf.fit(
    X_train,
    y_train
)

print("Random Forest training completed.")


# ------------------------------------------------
# 19. VALIDATION
# ------------------------------------------------

y_pred = rf.predict(X_test)


accuracy = accuracy_score(
    y_test,
    y_pred
)


f1 = f1_score(
    y_test,
    y_pred,
    pos_label=1
)


cm = confusion_matrix(
    y_test,
    y_pred
)


print("\n=============== VALIDATION ===============")

print(
    "Overall Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

print(
    "Tree F1 Score:",
    round(f1 * 100, 2),
    "%"
)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Non-tree",
            "Tree"
        ]
    )
)


# ------------------------------------------------
# 20. CLASSIFY COMPLETE SATELLITE IMAGE
# ------------------------------------------------

print("\nClassifying complete satellite image...")


# Convert:
#
# bands x rows x columns
#
# to:
#
# pixels x bands

all_pixels = features.reshape(
    features.shape[0],
    -1
).T


valid_flat = valid_pixels.reshape(-1)


classification = np.zeros(
    all_pixels.shape[0],
    dtype=np.uint8
)


# Only classify valid pixels
valid_indices = np.where(
    valid_flat
)[0]


print(
    "Pixels to classify:",
    len(valid_indices)
)


# Process in chunks to avoid RAM problems

CHUNK_SIZE = 500000


for start in range(
    0,
    len(valid_indices),
    CHUNK_SIZE
):

    end = min(
        start + CHUNK_SIZE,
        len(valid_indices)
    )

    idx = valid_indices[start:end]

    classification[idx] = rf.predict(
        all_pixels[idx]
    ).astype(np.uint8)


# ------------------------------------------------
# 21. RESHAPE CLASSIFICATION
# ------------------------------------------------

tree_map = classification.reshape(
    sat_height,
    sat_width
)


# ------------------------------------------------
# 22. SAVE TREE COVER MAP
# ------------------------------------------------

output_profile = sat_profile.copy()

output_profile.update({

    "count": 1,

    "dtype": "uint8",

    "compress": "lzw",

    "nodata": 0

})


with rasterio.open(
    OUTPUT_TREE,
    "w",
    **output_profile
) as dst:

    dst.write(
        tree_map,
        1
    )


print("\nTree cover map saved:")
print(OUTPUT_TREE)


# ------------------------------------------------
# 23. TREE AREA
# ------------------------------------------------
#
# Calculate approximate area from the satellite grid.
#
# This is most reliable when the image is in a projected
# CRS with metre units.
# ------------------------------------------------

try:

    pixel_width = abs(sat_transform.a)
    pixel_height = abs(sat_transform.e)

    pixel_area_m2 = (
        pixel_width *
        pixel_height
    )

    tree_pixels = np.sum(
        tree_map == 1
    )

    tree_area_ha = (
        tree_pixels *
        pixel_area_m2 /
        10000
    )

    total_area_ha = (
        np.sum(valid_pixels) *
        pixel_area_m2 /
        10000
    )

    print("\n=============== AREA ===============")

    print(
        "Tree pixels:",
        tree_pixels
    )

    print(
        "Tree area:",
        round(tree_area_ha, 2),
        "ha"
    )

    print(
        "Mapped area:",
        round(total_area_ha, 2),
        "ha"
    )

    if total_area_ha > 0:

        print(
            "Tree cover:",
            round(
                tree_area_ha /
                total_area_ha *
                100,
                2
            ),
            "%"
        )

except Exception as e:

    print(
        "Area calculation skipped:",
        e
    )


# ------------------------------------------------
# 24. VISUALIZE RESULTS
# ------------------------------------------------

fig = plt.figure(
    figsize=(18, 12)
)


# Satellite
ax1 = plt.subplot(2, 2, 1)

rgb = np.moveaxis(
    features,
    0,
    -1
)

# Simple percentile stretch
p2 = np.nanpercentile(
    rgb,
    2
)

p98 = np.nanpercentile(
    rgb,
    98
)

rgb_display = (
    rgb - p2
) / (
    p98 - p2 + 1e-6
)

rgb_display = np.clip(
    rgb_display,
    0,
    1
)

ax1.imshow(
    rgb_display
)

ax1.set_title(
    "Input Satellite Image"
)

ax1.axis("off")


# Canopy height
ax2 = plt.subplot(2, 2, 2)

im = ax2.imshow(
    canopy,
    cmap="viridis"
)

plt.colorbar(
    im,
    ax=ax2,
    label="Canopy Height (m)"
)

ax2.set_title(
    "Meta Canopy Height"
)

ax2.axis("off")


# Training reference
ax3 = plt.subplot(2, 2, 3)

reference_display = np.zeros(
    canopy.shape,
    dtype=np.uint8
)

reference_display[
    tree_reference &
    valid_pixels
] = 1

ax3.imshow(
    reference_display,
    cmap="Greens",
    vmin=0,
    vmax=1
)

ax3.set_title(
    "Tree Reference Mask (CH ≥ 4 m)"
)

ax3.axis("off")


# Classification
ax4 = plt.subplot(2, 2, 4)

ax4.imshow(
    tree_map,
    cmap="Greens",
    vmin=0,
    vmax=1
)

ax4.set_title(
    "Random Forest Tree Cover"
)

ax4.axis("off")


plt.tight_layout()

plt.show()


print("\n================================================")
print("PROCESS COMPLETED")
print("================================================")
print("Input :", SATELLITE_FILE)
print("Output:", OUTPUT_TREE)
print("================================================")